In [1]:
%pip install -q peft bitsandbytes accelerate qwen-vl-utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import ast
import torch
import pandas as pd

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from qwen_vl_utils import process_vision_info

os.environ["HF_HUB_DISABLE_XET"] = "1"

PROJECT_DIR = r"C:\SNU_2026"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "models", "qwen3vl4b_qlora_ordering")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"

print("TRAIN_CSV:", TRAIN_CSV)
print("TRAIN_IMAGE_DIR:", TRAIN_IMAGE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("cuda:", torch.cuda.is_available())

TRAIN_CSV: C:\SNU_2026\data\train.csv
TRAIN_IMAGE_DIR: C:\SNU_2026\data\train
OUTPUT_DIR: C:\SNU_2026\models\qwen3vl4b_qlora_ordering
cuda: True


In [3]:
# 2. 데이터 로드 및 작은 학습셋 만들기
train_df = pd.read_csv(TRAIN_CSV)

train_part, valid_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["Answer"],
)

train_small = train_part.sample(
    n=500,
    random_state=42,
).reset_index(drop=True)

valid_small = valid_part.sample(
    n=100,
    random_state=42,
).reset_index(drop=True)

print("train_small:", train_small.shape)
print("valid_small:", valid_small.shape)
display(train_small.head())

train_small: (500, 8)
valid_small: (100, 8)


,Id,Input_1,Input_2,Input_3,Input_4,Sentence,Answer,No_ordering
0,3yKoYW,3yKoYW_adr.jpg,3yKoYW_ato.jpg,3yKoYW_awh.jpg,3yKoYW_mwp.jpg,The pasta dish is plated and served at the tab...,"[2, 3, 4, 1]",False
1,9Oybn0,9Oybn0_cgm.jpg,9Oybn0_dvn.jpg,9Oybn0_qam.jpg,9Oybn0_qwi.jpg,"A girl pets a horse as the camera pans right, ...","[1, 3, 2, 4]",False
2,JhaSzY,JhaSzY_leg.jpg,JhaSzY_mzi.jpg,JhaSzY_nkd.jpg,JhaSzY_syr.jpg,The scene opens with a snow castle at the Wint...,"[1, 4, 3, 2]",False
3,Z6sWtv,Z6sWtv_dsw.jpg,Z6sWtv_pen.jpg,Z6sWtv_vbp.jpg,Z6sWtv_zsh.jpg,The camera pans left from the technician's han...,"[1, 2, 3, 4]",True
4,3Sc15Q,3Sc15Q_nsw.jpg,3Sc15Q_sff.jpg,3Sc15Q_soz.jpg,3Sc15Q_tgl.jpg,The individual begins to turn and point to the...,"[4, 2, 1, 3]",False


In [4]:
import ast
import random

def parse_answer(answer_text):
    if isinstance(answer_text, list):
        return answer_text
    return ast.literal_eval(str(answer_text))


def submission_answer_to_order(answer):
    order = [0] * 4

    for image_idx, position in enumerate(answer):
        order[position - 1] = image_idx + 1

    return order

In [5]:
#QLoRA 모델 로드
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    min_pixels=128 * 28 * 28,
    max_pixels=448 * 28 * 28,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\lyh30\anaconda3\envs\snu2026\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\lyh30\anaconda3\envs\snu2026\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\lyh30\anaconda3\envs\snu2026\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
W0721 15:14:11.497000 4328 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

c:\Users\lyh30\anaconda3\envs\snu2026\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 33,030,144 || all params: 4,470,845,952 || trainable%: 0.7388


In [9]:
#학습용 Dataset 만들기
class QwenOrderingDataset(torch.utils.data.Dataset):
    def __init__(self, df, image_dir, processor, shuffle_images=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.processor = processor
        self.shuffle_images = shuffle_images

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_files = [
            row["Input_1"],
            row["Input_2"],
            row["Input_3"],
            row["Input_4"],
        ]

        # 원래 Input_1~Input_4 기준의 실제 시간 순서
        true_answer = parse_answer(row["Answer"])
        true_order_original = submission_answer_to_order(true_answer)

        # 학습할 때는 이미지 표시 순서를 랜덤하게 섞습니다.
        if self.shuffle_images:
            perm = list(range(4))
            random.shuffle(perm)
        else:
            perm = [0, 1, 2, 3]

        content = []

        for display_idx, original_idx in enumerate(perm):
            img_file = img_files[original_idx]
            img_path = os.path.join(
                self.image_dir,
                str(row["Id"]),
                str(img_file),
            )

            content.append({
                "type": "image",
                "image": img_path,
            })

            content.append({
                "type": "text",
                "text": f"\nImage {display_idx + 1}\n",
            })

        sentence = str(row["Sentence"])

        prompt = f"""
The 4 images above are shuffled video frames.
The displayed image order is random and must NOT be trusted.

Sentence:
{sentence}

Determine the correct chronological order of the displayed images.

Important:
- Do NOT assume Image 1, Image 2, Image 3, Image 4 are already in order.
- Compare visual changes carefully.
- Return ONLY a Python list of 4 integers.
- Each integer must be one of 1, 2, 3, 4.
- Use each integer exactly once.

Example output:
[2, 1, 4, 3]
""".strip()

        content.append({
            "type": "text",
            "text": prompt,
        })

        user_messages = [
            {
                "role": "user",
                "content": content,
            }
        ]

        # 핵심:
        # 원래 이미지 번호 기준 정답을,
        # 현재 화면에 표시된 Image 번호 기준 정답으로 변환합니다.
        true_order_display = []

        for original_image_num in true_order_original:
            original_idx = original_image_num - 1
            display_idx = perm.index(original_idx)
            true_order_display.append(display_idx + 1)

        target_text = str(true_order_display)

        full_messages = user_messages + [
            {
                "role": "assistant",
                "content": target_text,
            }
        ]

        prompt_text = self.processor.apply_chat_template(
            user_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        full_text = self.processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        image_inputs, video_inputs = process_vision_info(user_messages)

        full_inputs = self.processor(
            text=[full_text],
            images=image_inputs,
            videos=video_inputs,
            padding=False,
            return_tensors="pt",
        )

        prompt_inputs = self.processor(
            text=[prompt_text],
            images=image_inputs,
            videos=video_inputs,
            padding=False,
            return_tensors="pt",
        )

        labels = full_inputs["input_ids"].clone()
        prompt_len = prompt_inputs["input_ids"].shape[1]
        labels[:, :prompt_len] = -100

        full_inputs["labels"] = labels

        return full_inputs

In [10]:
#collate 함수
def qwen_collate_fn(features):
    return features[0]

In [11]:
train_dataset = QwenOrderingDataset(
    train_small,
    TRAIN_IMAGE_DIR,
    processor,
    shuffle_images=True,   # 학습 때는 섞기
)

valid_dataset = QwenOrderingDataset(
    valid_small,
    TRAIN_IMAGE_DIR,
    processor,
    shuffle_images=False,  # validation loss 계산용은 고정
)

In [14]:
#학습 실행
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    logging_steps=10,

    eval_strategy="steps",
    eval_steps=100,

    save_steps=100,
    save_total_limit=1,

    fp16=True,
    bf16=False,

    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=qwen_collate_fn,
)

trainer.train()

c:\Users\lyh30\anaconda3\envs\snu2026\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\lyh30\anaconda3\envs\snu2026\Lib\site-packages\torch\utils\checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [13]:
#LoRA adapter 저장
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("saved:", OUTPUT_DIR)

NameError: name 'trainer' is not defined

In [33]:
def make_qwen_messages_for_eval(row, image_dir):
    img_files = [
        row["Input_1"],
        row["Input_2"],
        row["Input_3"],
        row["Input_4"],
    ]

    content = []

    for i, img_file in enumerate(img_files):
        img_path = os.path.join(
            image_dir,
            str(row["Id"]),
            str(img_file),
        )

        content.append({
            "type": "image",
            "image": img_path,
        })

        content.append({
            "type": "text",
            "text": f"\nImage {i + 1}\n",
        })

    sentence = str(row["Sentence"])

    prompt = f"""
The 4 images above are shuffled video frames.
They are NOT guaranteed to be in chronological order.

Sentence:
{sentence}

Determine the correct chronological order of the images.

Return ONLY a Python list of 4 integers.
Each integer must be one of 1, 2, 3, 4.
Use each integer exactly once.

Example output:
[2, 1, 4, 3]
""".strip()

    content.append({
        "type": "text",
        "text": prompt,
    })

    return [
        {
            "role": "user",
            "content": content,
        }
    ]

In [15]:
import ast
import re

def parse_order_output(output_text):
    """
    모델 출력 문자열에서 [1, 2, 3, 4] 형태의 순서 리스트를 추출합니다.

    예:
    raw output = "The answer is [4, 2, 1, 3]"
    return = [4, 2, 1, 3]
    """
    output_text = str(output_text).strip()

    # [4, 2, 1, 3] 같은 대괄호 리스트를 먼저 찾습니다.
    match = re.search(r"\[[^\]]+\]", output_text)

    if match:
        try:
            result = ast.literal_eval(match.group())

            if (
                isinstance(result, list)
                and len(result) == 4
                and sorted(result) == [1, 2, 3, 4]
            ):
                return result

        except Exception:
            pass

    # 혹시 모델이 대괄호 없이 "4, 2, 1, 3"처럼 출력한 경우도 처리합니다.
    numbers = [int(x) for x in re.findall(r"\b[1-4]\b", output_text)]

    if len(numbers) >= 4:
        result = numbers[:4]

        if sorted(result) == [1, 2, 3, 4]:
            return result

    # 파싱 실패 시 기본값
    return [1, 2, 3, 4]

In [16]:
#추론함수
def predict_qwen_order_finetuned(row, image_dir):
    messages = make_qwen_messages_for_eval(row, image_dir)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            use_cache=True,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    raw_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    pred_order = parse_order_output(raw_text)

    return pred_order, raw_text

In [18]:
def order_to_submission_answer(order):
    """
    모델이 예측한 chronological order를 대회 제출 Answer 형식으로 변환합니다.

    모델 출력 예:
    order = [4, 2, 1, 3]

    의미:
    1번째 시간 = Image 4
    2번째 시간 = Image 2
    3번째 시간 = Image 1
    4번째 시간 = Image 3

    대회 Answer 형식:
    answer = [3, 2, 4, 1]

    의미:
    Image 1은 3번째
    Image 2는 2번째
    Image 3은 4번째
    Image 4는 1번째
    """
    answer = [0] * 4

    for position, image_num in enumerate(order):
        answer[image_num - 1] = position + 1

    return answer

In [19]:
correct = 0
total = 0
identity_count = 0
bad_outputs = []

for idx, row in tqdm(valid_small.iterrows(), total=len(valid_small)):
    pred_order, raw_text = predict_qwen_order_finetuned(
        row,
        TRAIN_IMAGE_DIR,
    )

    pred_answer = order_to_submission_answer(pred_order)
    true_answer = parse_answer(row["Answer"])

    if pred_answer == true_answer:
        correct += 1

    if pred_order == [1, 2, 3, 4]:
        identity_count += 1

    if sorted(pred_order) != [1, 2, 3, 4]:
        bad_outputs.append((row["Id"], raw_text))

    total += 1

accuracy = correct / total
identity_ratio = identity_count / total

print("finetuned valid_small accuracy:", accuracy)
print("identity ratio:", identity_ratio)
print("bad outputs:", len(bad_outputs))

  0%|          | 0/20 [00:00<?, ?it/s]

c:\Users\lyh30\anaconda3\envs\snu2026\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


finetuned valid_small accuracy: 0.2
identity ratio: 0.85
bad outputs: 0


## 4. 결과 저장 (Submission)

### 제출 파일 형식 (submission.csv)

제출 파일은 반드시 아래 형식을 따라야 합니다:

| 컬럼 | 타입 | 설명 | 예시 |
|------|------|------|------|
| `Id` | string | 테스트 샘플 고유 ID | `WI8o3F` |
| `Answer` | string | 정답 순서대로 다시 배열하였을 때 각 프레임의 위치 (Python 리스트 형태의 문자열) | `[3, 1, 4, 2]` |

#### 올바른 예시

```csv
Id,Answer
WI8o3F,[3, 1, 4, 2]
Ae7zLm,[1, 2, 3, 4]
...


In [ ]:
#제출 파일 저장
submission_df = test_df[["Id"]].merge(
    submission_df,
    on="Id",
    how="left",
)

submission_df["Answer"] = submission_df["Answer"].fillna(str([1, 2, 3, 4]))

submission_df.to_csv(SUBMISSION_PATH, index=False)

print("saved:", SUBMISSION_PATH)
print("submission shape:", submission_df.shape)

display(submission_df.head())